In [8]:
# Registration Number: 23MID0319

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
dataset_url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(dataset_url)

print("--- First 5 Rows of the Dataset ---")
print(df.head())

print(f"\nDataset Dimensions: {df.shape}")
print("\nAvailable Features in Dataset:")
print(df.columns)

print("\n--- Data Structure Info ---")
df.info()
print("\nCounting Missing Values per Column:")
print(df.isnull().sum())

print("\n--- Summary Statistics ---")
print(df.describe())

target_var = "median_house_value"
X = df.drop(columns=[target_var])
y = df[target_var]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

num_cols = X_train.select_dtypes(include=np.number).columns
cat_cols = X_train.select_dtypes(exclude=np.number).columns

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

print("\n[Status] Data preprocessing pipeline built successfully.")

# 4. REGRESSION MODELING & EVALUATION
def print_metrics(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n=== {model_name} Results ===")
    print(f"Mean Absolute Error (MAE)   : {mae:.4f}")
    print(f"Root Mean Sq. Error (RMSE)  : {rmse:.4f}")
    print(f"R-squared Score (R2)        : {r2:.4f}")

# --- MODEL 1: SIMPLE LINEAR REGRESSION (23MID0319) ---
simple_model = LinearRegression()
simple_model.fit(X_train[["median_income"]], y_train)
simple_pred = simple_model.predict(X_test[["median_income"]])

print_metrics("Simple Linear Regression (Baseline)", y_test, simple_pred)

# --- MODEL 2: MULTIPLE LINEAR REGRESSION (23MID0319) ---
multiple_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LinearRegression())
])
multiple_model.fit(X_train, y_train)
multiple_pred = multiple_model.predict(X_test)

print_metrics("Multiple Linear Regression", y_test, multiple_pred)

# --- MODEL 3: POLYNOMIAL REGRESSION (23MID0319) ---
poly_features = ["median_income", "housing_median_age", "total_rooms"]

poly_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])
poly_model.fit(X_train[poly_features], y_train)
poly_pred = poly_model.predict(X_test[poly_features])

print_metrics("Polynomial Regression (Degree=2)", y_test, poly_pred)

# --- MODEL 4: RIDGE REGRESSION (23MID0319) ---
ridge_model = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge(alpha=1.0))
])
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

print_metrics("Ridge Regression (L2 Regularized)", y_test, ridge_pred)

--- First 5 Rows of the Dataset ---
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  

Dataset Dimensions: (20640, 10)

Avail